### Extract and Transform AHRQ Survey data, filtering for SDOH topics.
Extract and clean AHRQ survey data. Save a dictionary to be used for LangChain SQLDatabase tool.  
Provide unction out_AHRQCountySDOH() to return dataframe. See https://www.ahrq.gov/sdoh/data-analytics/sdoh-data.html and https://www.ahrq.gov/sites/default/files/wysiwyg/sdoh/SDOH-Data-Sources-Documentation-v1-Final.pdf

Input parameters:
- parm_AHRQCountySDOH_years: list of survey years
- parm_AHRQCountySDOH_surveys: list of surveys to extract
- parm_AHRQCountySDOH_questions: CDCW and CHR have some SDOH questions which could be used (add these)

Typical surveys to use include:

["ACS", "AHA", "AMFAR", "CCBP", "CDCSVI", "CEN", "CRDC", "EPAA", "FARA", "FEA", "HHC", "HIFLD", "HRSA", "MHSVI", "MP", "NCHS", "NEPHTN", "NHC", "NOAAC", "NOAAS", "POS", "SAHIE", "SAIPE", "SEDA"]

These surveys are health outcomes and would not be used as SDOH:

    AHRF, CDCA, CDCAP, CDCP, CDCW, CHR and MGV

    
However, CDCW and CHR have some SDOH questions which could be used in the list of questions:

["CDCW_INJURY_DTH_RATE", "CDCW_TRANSPORT_DTH_RATE", "CDCW_SELFHARM_DTH_RATE", "CDCW_ASSAULT_DTH_RATE", "CHR_TOT_MENTAL_PROV", "CHR_MENTAL_PROV_RATE", "CHR_SEGREG_BLACK", "CHR_PCT_ALCOHOL_DRIV_DEATH", "CHR_PCT_EXCESS_DRINK", "CHR_PCT_FOOD", "CHR_SEGREG_BLACK", "CHR_SEGREG_NON_WHITE"]

Note 2018 has some behavioral health questions which could be used:

["CDCP_NO_PHY_ACTV_ADULT_A", "CDCP_NO_PHY_ACTV_ADULT_C", "CDCP_SLEEP_LESS7HR_ADULT_A", "CDCP_SLEEP_LESS7HR_ADULT_C"]
    
CAF are County Adjacent FIPS codes which could be considered.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import requests
import re
import pickle
import sys
import time
from io import BytesIO
from typing import Optional, Sequence
# for imports from agents
sys.path.append('../agents/tools')

### Parms
From calling Notebook

In [2]:
# parm_AHRQCountySDOH_years = ['2015', '2016']
# parm_AHRQCountySDOH_surveys = ["ACS", "AHA", "AMFAR", "CAF", "CCBP", "CDCSVI", "CEN", "CRDC", "EPAA", "FARA", "FEA", "HHC", "HIFLD", "HRSA", "MHSVI", "MP", "NCHS", "NEPHTN", "NHC", "NOAAS", "POS", "SAHIE", "SAIPE", "SEDA"]
# parm_AHRQCountySDOH_questions = ["CDCW_INJURY_DTH_RATE", "CDCW_TRANSPORT_DTH_RATE", "CDCW_SELFHARM_DTH_RATE", "CDCW_ASSAULT_DTH_RATE", "CHR_TOT_MENTAL_PROV", "CHR_MENTAL_PROV_RATE", "CHR_SEGREG_BLACK", "CHR_PCT_ALCOHOL_DRIV_DEATH", "CHR_PCT_EXCESS_DRINK", "CHR_PCT_FOOD", "CHR_SEGREG_BLACK", "CHR_SEGREG_NON_WHITE"]

### Download Method
If we are running in a container, download can get CloudFront 403 meaning the site is blocking “non-browser” clients. Fix it by sending browser-like headers.

In [3]:
def download_excel_with_browser_headers(url: str, out_path: str = None, session: requests.Session = None, timeout: int = 30) -> bytes:
    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"),
        "Accept": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        # "Referer": "https://www.ahrq.gov/sdoh/index.html",
    }
    s = session or requests.Session()
    r = s.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    content = r.content
    if out_path:
        with open(out_path, "wb") as f:
            f.write(content)
    return content

### Download

In [4]:
# AHRQCountySDOH_year = parm_AHRQCountySDOH_years.pop(0)
# url = 'https://www.ahrq.gov/sites/default/files/wysiwyg/sdoh/SDOH_'+  AHRQCountySDOH_year +'_COUNTY_1_0.xlsx'
# out_path = download_excel_with_browser_headers(url, "./ahrq"+ AHRQCountySDOH_year +".xlsx")
# dfAHRQCountySDOH = pd.read_excel(out_path, sheet_name="Data", engine="openpyxl")
# dfAHRQCountySDOH = dfAHRQCountySDOH.drop(dfAHRQCountySDOH.columns[[ 0,1,2,5,6 ]],axis = 1)
# dfAHRQCountySDOH['YEAR'] =  AHRQCountySDOH_year

# # Import the remaining excel files and append to dataframe
# for AHRQCountySDOH_year in parm_AHRQCountySDOH_years:
#     out_path = download_excel_with_browser_headers(url, "./ahrq"+ AHRQCountySDOH_year +".xlsx")
#     dfAHRQCountySDOHnext = pd.read_excel(out_path, sheet_name="Data", engine="openpyxl")
#     dfAHRQCountySDOHnext = dfAHRQCountySDOHnext.drop(dfAHRQCountySDOHnext.columns[[ 0,1,2,5,6 ]],axis = 1)
#     dfAHRQCountySDOHnext['YEAR'] =  AHRQCountySDOH_year
#     dfAHRQCountySDOH = pd.concat([dfAHRQCountySDOH, dfAHRQCountySDOHnext], ignore_index=True)

In [5]:
df_list = []
failed_years = []
session = requests.Session()

for yr in parm_AHRQCountySDOH_years:
    url = f'https://www.ahrq.gov/sites/default/files/wysiwyg/sdoh/SDOH_{yr}_COUNTY_1_0.xlsx'
    out_path = f"./ahrq{yr}.xlsx"
    try:
        # small retry loop for transient failures
        attempts = 0
        while True:
            attempts += 1
            try:
                excel_bytes = download_excel_with_browser_headers(url, out_path=out_path, session=session)
                break
            except requests.HTTPError as e:
                if attempts >= 3:
                    raise
                time.sleep(1 * attempts)
        # read directly from bytes (avoids needing to rely on disk if you want)
        df = pd.read_excel(pd.io.common.BytesIO(excel_bytes), sheet_name="Data", engine="openpyxl")
        # replicate your column-dropping and YEAR assignment
        df = df.drop(df.columns[[0, 1, 2, 5, 6]], axis=1)
        df['YEAR'] = yr
        df_list.append(df)
        print(f"Loaded year {yr} ({len(df)} rows).")
    except Exception as e:
        failed_years.append((yr, str(e)))
        print(f"Failed to load year {yr}: {e}")

# concatenate all successfully loaded years
if df_list:
    dfAHRQCountySDOH = pd.concat(df_list, ignore_index=True)
else:
    dfAHRQCountySDOH = pd.DataFrame()  # empty fallback

if failed_years:
    print("Some years failed to load:", failed_years)

NameError: name 'parm_AHRQCountySDOH_years' is not defined

### Clean data
Convert to numeric, forcing non-convertible values to NaN and remove county from names.

In [ ]:
for col in dfAHRQCountySDOH.columns[3:]:
    dfAHRQCountySDOH[col] = pd.to_numeric(dfAHRQCountySDOH[col], errors='coerce')
dfAHRQCountySDOH['COUNTY'] = dfAHRQCountySDOH['COUNTY'].str.replace(' County','')

### Filter to SDOH surverys

In [ ]:
dfAHRQCountySDOHred = dfAHRQCountySDOH[["STATE", "COUNTY", "YEAR"]] 

dfAHRQCountySDOHsel = dfAHRQCountySDOH[dfAHRQCountySDOH.columns[pd.Series(dfAHRQCountySDOH.columns).str.startswith(tuple(parm_AHRQCountySDOH_surveys))]]
dfAHRQCountySDOHred2 = pd.concat([dfAHRQCountySDOHred, dfAHRQCountySDOHsel], axis=1)

dfAHRQCountySDOHsel = dfAHRQCountySDOH[parm_AHRQCountySDOH_questions]
dfAHRQCountySDOHred = pd.concat([dfAHRQCountySDOHred2, dfAHRQCountySDOHsel], axis=1)
dfAHRQCountySDOHnew = dfAHRQCountySDOHred

### Missing values
Commented out, keep all columns for now.

Remove columns missing more than 30%.  Then impute missing data using KNN imputation.

In [ ]:
# # drop columns that are more than 30% null
# dfAHRQCountySDOHnew = dfAHRQCountySDOHnew.dropna(axis = 1, thresh=len(dfAHRQCountySDOHnew)*.7)

# from sklearn.impute import KNNImputer
# imputer = KNNImputer(n_neighbors=5, weights="uniform")
# dfAHRQCountySDOHnew.iloc[:,3::] = imputer.fit_transform(dfAHRQCountySDOHnew.iloc[:,3::])

### Method to get SDOH Code Book, filter for SDOH data and build dictionary of column descriptors

In [ ]:
def load_sdoh_codebook_as_custom_table_info(
    codebook_url: str,
    sheet_name: str,
    table_name: str,
    table_description: str = "AHRQ SDOH dataset",
    out_path: str = "./ahrq_codebook.xlsx",
    allowed_prefixes: Sequence[str] | None = None,
    allowed_questions: Sequence[str] | None = None,
    verbose: bool = True
) -> dict[str, dict]:
    """
    Parameters
    ----------
    codebook_url : str
        HTTPS URL to the Codebook file.
    sheet_name : str
        Sheet name to load from the Excel workbook (required).
    table_name : str
        Table name as it exists in the database.
    table_description : str, optional
        Description of the table contents.
    out_path : str, optional
        Local file path to save the codebook.
    allowed_prefixes : list[str] or None
        Keep variables that start with any of these prefixes (case-insensitive).
    allowed_questions : list[str] or None
        Keep these exact variable names as well (case-insensitive).
    verbose : bool
        Print summary information.

    Returns
    dict[str, dict]
        custom_table_info structure, e.g.
        {
            "table_name": {
                "table_description": "...",
                "columns": {
                    "col1": "Human description",
                    "col2": "Human description"
                }
            }
        }
    """
 # Download (may return bytes or be a path written to out_path)
    file_or_bytes = download_excel_with_browser_headers(codebook_url, out_path=out_path)

    # Normalize to file-like object if bytes were returned
    if isinstance(file_or_bytes, (bytes, bytearray)):
        excel_source = BytesIO(file_or_bytes)
    else:
        excel_source = file_or_bytes  # file path or file-like object; pandas accepts it

    # Read only the requested sheet (let pandas raise if sheet not present)
    df = pd.read_excel(excel_source, sheet_name=sheet_name, engine="openpyxl")

    # Find variable and label columns
    var_col = next(
        (c for c in df.columns if re.search(r'\b(var(iable)?(\s*name)?|short\s*name)\b', str(c), re.I)),
        None
    )
    label_col = next(
        (c for c in df.columns if re.search(r'\b(label|description|long\s*name|title)\b', str(c), re.I)),
        None
    )

    if var_col is None or label_col is None:
        raise ValueError(
            f"Could not identify variable/label columns in sheet '{sheet_name}'. Columns seen: {list(df.columns)}"
        )

    # Build a small dataframe of vars -> labels (clean NA)
    vars_df = df[[var_col, label_col]].dropna(subset=[var_col]).copy()
    vars_df.columns = ["variable", "label"]
    vars_df["variable"] = vars_df["variable"].astype(str).str.strip()
    vars_df["label"] = vars_df["label"].astype(str).str.strip().fillna("")

    # Prepare filters (case-insensitive)
    kept_mask = pd.Series(False, index=vars_df.index)

    if allowed_prefixes:
        prefixes_tuple = tuple(str(p).strip().lower() for p in allowed_prefixes if str(p).strip())
        if prefixes_tuple:
            kept_mask = kept_mask | vars_df["variable"].str.lower().str.startswith(prefixes_tuple)

    if allowed_questions:
        qset = {str(q).strip().lower() for q in allowed_questions if str(q).strip()}
        if qset:
            kept_mask = kept_mask | vars_df["variable"].str.lower().isin(qset)

    # If no filters provided, keep all
    if not allowed_prefixes and not allowed_questions:
        kept_mask = pd.Series(True, index=vars_df.index)

    kept_df = vars_df[kept_mask].copy()
    dropped_df = vars_df[~kept_mask].copy()

    if verbose:
        print(f"Codebook sheet '{sheet_name}': total vars={len(vars_df)}, kept={len(kept_df)}, dropped={len(dropped_df)}")
        if len(dropped_df) and len(dropped_df) <= 40:
            print("Dropped sample:", dropped_df["variable"].tolist()[:20])

    # Build columns dict in the expected format
    columns = {row["variable"]: row["label"] for _, row in kept_df.iterrows()}

    custom_table_info = {
        table_name: {
            "table_description": table_description,
            "columns": columns
        }
    }
    return custom_table_info


## Method to normalize column names

In [ ]:
def normalize_colname(name: str) -> str:
    """
    1) lower-case
    2) replace any non-alphanumeric character with underscore
    3) collapse multiple underscores
    4) strip leading/trailing underscores
    5) if starts with digit -> prefix with 'col_'
    6) special-case a 'year' column -> rename to 'year_col' to avoid reserved-word collisions
    """
    if name is None:
        return name
    s = str(name).lower()
    # replace non-alphanumeric with underscore
    s = re.sub(r'[^a-z0-9]', '_', s)
    # collapse multiple underscores
    s = re.sub(r'_+', '_', s)
    s = s.strip('_')
    # if empty after cleaning
    if not s:
        s = 'col'
    # prefix if starts with digit
    if re.match(r'^[0-9]', s):
        s = 'col_' + s
    # avoid a bare "year" column name that can sometimes be problematic
    if s == 'year':
        s = 'year'
    return s

### Method to normalize names in the mapping

In [ ]:
def normalize_mapping_columns(mapping: dict) -> dict:
    """
    Normalize all column names in the mapping using normalize_colname().
    """
    normalized_mapping = {}
    for table, tinfo in mapping.items():
        normalized_cols = {normalize_colname(col): desc for col, desc in tinfo.get("columns", {}).items()}
        normalized_mapping[table] = {
            "table_description": tinfo.get("table_description", ""),
            "columns": normalized_cols
        }
    return normalized_mapping

### Build Database Dictionary and also save as csv for reference
Get column descriptors, normalize columns and create dictionary.  Note this is simple as it assumes
we are rebuilding the dictionary each time.

In [ ]:
from dictionary_tool import DictionaryLocalTool

mapping = load_sdoh_codebook_as_custom_table_info(
    "https://www.ahrq.gov/sites/default/files/wysiwyg/sdoh/SDOH_2020_Codebook_1_0.xlsx",
    "County",
    "sdoh_surveys",
    "AHRQ SDOH Surveys",
    "./ahrq_codebook.xlsx",
    allowed_prefixes=parm_AHRQCountySDOH_surveys,
    allowed_questions=parm_AHRQCountySDOH_questions
)

mapping = normalize_mapping_columns(mapping)

dt = DictionaryLocalTool(persist_dir="../data", model_name="all-MiniLM-L6-v2", search_k=6)
dt.create_index(mapping, dedupe=True, overwrite=True )  # or dt.rebuild_index(mapping)

# Convert mapping to a flat CSV format
rows = []
for table_name, table_info in mapping.items():
    table_desc = table_info.get("table_description", "")
    for col_name, col_desc in table_info.get("columns", {}).items():
        rows.append({
            "table_name": table_name,
            "table_description": table_desc,
            "column_name": col_name,
            "column_description": col_desc
        })

# Create DataFrame and write to CSV
df_dictionary = pd.DataFrame(rows)
df_dictionary.to_csv("dictionary.csv", index=False)
print(f"Dictionary saved to dictionary.csv with {len(df_dictionary)} rows")

### Load function
Function for calling notebook to get dataframe

In [ ]:
# remove formatting issues by converting YEAR to int
dfAHRQCountySDOHnew["YEAR"] = pd.to_numeric(dfAHRQCountySDOHnew["YEAR"])
def out_AHRQCountySDOH():
   return dfAHRQCountySDOHnew